# 05 — RCA Graph Demo

Demonstrates both rule-based graph RCA and GNN RCA on a synthetic incident.

1. Load a ground-truth topology graph
2. Simulate a realistic alert sequence
3. Run rule-based RCA
4. Run GNN RCA (if torch-geometric is installed)
5. Compare predictions against the known root cause

In [ ]:
import sys, json, random
sys.path.insert(0, '..')

from pathlib import Path
import networkx as nx
import matplotlib.pyplot as plt

from src.rca.alert_simulator  import simulate_incident
from src.rca.graph_rca        import run_rca
from src.utils.visualization  import plot_topology_graph, plot_rca_scores

GRAPH_DIR = Path('../data_generator/infragraph_dataset/graphs/train')
graph_files = sorted(GRAPH_DIR.glob('*.json'))
gf = random.choice(graph_files)
print('Loaded:', gf.name)

In [ ]:
raw = json.loads(gf.read_text())

G = nx.Graph()
for n in raw['nodes']:
    G.add_node(n['id'], class_name=n['type'], label=n['id'], bbox=n.get('bbox', [0,0,0,0]))
for e in raw['edges']:
    G.add_edge(e['source'], e['target'])

print(f'Nodes: {G.number_of_nodes()}  Edges: {G.number_of_edges()}')
fig = plot_topology_graph(G, title=f'Topology — {gf.stem}')
plt.show()

In [ ]:
incident = simulate_incident(G, seed=random.randint(0, 9999))
true_root = incident['root_cause']
print(f'True root cause : {true_root} ({incident["root_cause_class"]})')
print(f'Alerts fired    : {len(incident["alerts"])}')
for a in incident['alerts']:
    print(f"  +{a['time_offset_min']:2d}m  [{a['severity']:<8}]  {a['node']} — {a['alert_type']}")

In [ ]:
candidates = run_rca(G, incident['alerts'], top_k=3)
print('\nRule-based RCA — top candidates:')
for rank, c in enumerate(candidates, 1):
    marker = '✓' if c['node'] == true_root else ' '
    print(f'  {rank}. [{marker}] {c["node"]} ({c["class_name"]})  score={c["score"]}')

fig = plot_rca_scores(candidates, title='Rule-based RCA Scores')
plt.show()

In [ ]:
# Highlight predicted root cause on topology
top1 = candidates[0]['node'] if candidates else None
fig  = plot_topology_graph(
    G,
    title=f'RCA Result — predicted: {top1}  |  true: {true_root}',
    highlight_nodes=[top1, true_root],
)
plt.show()

In [ ]:
# GNN RCA (requires torch-geometric)
try:
    from src.rca.gnn_rca import GATRCAModel, predict_root_cause
    import torch

    model = GATRCAModel()
    # NOTE: load trained weights here: model.load_state_dict(torch.load('...'))
    # Using untrained model for demo — predictions are random.
    gnn_candidates = predict_root_cause(model, G, incident['alerts'], top_k=3)
    print('GNN RCA — top candidates (untrained):')
    for rank, c in enumerate(gnn_candidates, 1):
        marker = '✓' if c['node'] == true_root else ' '
        print(f'  {rank}. [{marker}] {c["node"]} ({c["class_name"]})  score={c["score"]:.4f}')
except ImportError as e:
    print(f'GNN RCA unavailable: {e}')
    print('Install with: pip install torch-geometric')